# 06. SWE mode

Финальный слой: отдельный граф для issue-resolution. Он строже относится к воспроизведению, локализации, regression coverage и подтверждению записей.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import httpx
from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openrouter:openai/gpt-oss-120b:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)


In [ ]:
BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.

If local shell execution is unavailable, use filesystem tools and clearly state
which verification would require shell access.
"""

In [ ]:
SWE_PROMPT = BASE_PROMPT + """\

You are running in SWE mode. Optimize for issue resolution:
- reproduce or characterize the failure first;
- localize the root cause before patching;
- add regression coverage where practical;
- run the narrowest useful verification before broad checks;
- keep a clear chain from issue to patch to test.
"""

In [ ]:
def workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", ".")).expanduser().resolve()


def backend():
    """Filesystem by default; local shell only when explicitly enabled."""
    root_dir = workspace_root()

    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") != "1":
        from deepagents.backends import FilesystemBackend

        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)

    from deepagents.backends import LocalShellBackend

    return LocalShellBackend(
        root_dir=root_dir,
        virtual_mode=True,
        inherit_env=True,
        timeout=120,
        max_output_bytes=80_000,
    )

In [ ]:
SUBAGENTS = [
    {
        "name": "repo-researcher",
        "description": "Map repository structure, APIs, tests, and likely change locations.",
        "system_prompt": (
            "You research codebases. Inspect files, identify relevant modules, "
            "and return concise findings with file paths and rationale."
        ),
    },
    {
        "name": "reviewer",
        "description": "Review proposed patches for bugs, missing tests, and regressions.",
        "system_prompt": (
            "You are a code reviewer. Focus on correctness, edge cases, tests, "
            "security, and maintainability. Be specific and concise."
        ),
    },
]

In [ ]:
from connectors import CONNECTOR_TOOLS

swe_agent = create_deep_agent(
    model=model_name(),
    tools=CONNECTOR_TOOLS,
    system_prompt=SWE_PROMPT,
    subagents=SUBAGENTS,
    skills=["/skills/swe-resolution"],
    memory=["/AGENTS.md"],
    backend=backend(),
    interrupt_on={"execute": True, "write_file": True, "edit_file": True},
)

In [ ]:
print(type(swe_agent).__name__)

Demo prompt: "Inspect the project shape test and explain what behavior it protects. If you change code, follow SWE mode".